<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/notebooks/medgemma_in-context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.9 MB/s eta 0:00:00


## Download data

In [2]:
!wget -nc -q "https://drive.google.com/uc?export=download&id=1w0_XjEl4gCrDWxFnsJHrhsVEGTiF3Ayh" -O trainingData.zip
!unzip -q trainingData.zip

## Collect drug data

In [3]:
import json
import glob
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Data Loading Logic
def load_drug_data(directory):
    all_data = []
    file_paths = glob.glob(os.path.join(directory, "*.json"))
    for path in file_paths:
        with open(path, 'r') as f:
            try:
                item = json.load(f)
                all_data.append(item)
            except: continue
    return all_data

drug_data = load_drug_data("/content/trainingFilesConverted/")

In [4]:
drug_data[0]

{'drug': 'Xarelto',
 'text': "WARNING: (A) PREMATURE DISCONTINUATION OF XARELTO INCREASES THE RISK OF THROMBOTIC EVENTS, (B) SPINAL/EPIDURAL HEMATOMA \n\n WARNING: (A) PREMATURE DISCONTINUATION OF XARELTO INCREASES THE RISK OF THROMBOTIC EVENTS, (B) SPINAL/EPIDURAL HEMATOMA \n\n See full prescribing information for complete boxed warning \n\n (A) Premature discontinuation of XARELTO increases the risk of thrombotic events \n\n Premature discontinuation of any oral anticoagulant, including XARELTO, increases the risk of thrombotic events. To reduce this risk, consider coverage with another anticoagulant if XARELTO is discontinued for a reason other than pathological bleeding or completion of a course of therapy ( 2.3 , 2.8 , 5.1 , 14.1 ). \n\n (B) Spinal/epidural hematoma \n\n Epidural or spinal hematomas have occurred in patients treated with XARELTO who are receiving neuraxial anesthesia or undergoing spinal puncture. These hematomas may result in long-term or permanent paralysis ( 5.

## Get model and tokenizer

In [5]:
model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Build in-context prompt

In [14]:
from IPython.display import display, Markdown

def run_in_context_extraction(target_drug, target_text, shots=2):
    # Select examples for the context
    context_examples = [ex for ex in drug_data if ex['drug'] != target_drug][:shots]

    # Build the prompt string
    sys_prompt = "You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.\n\n"

    output_input = ""
    formatted_input = "" # https://ai.google.dev/gemma/docs/core/prompt-structure
    for i, ex in enumerate(context_examples):
        output_input += "<start_of_turn>user\n"
        formatted_input += "<start_of_turn>user\n"

        if i == 0:
            output_input += sys_prompt
            formatted_input += sys_prompt

        output_input += f"Based on the following prescribing information, provide the drug-drug interactions for the drug, {ex['drug']}, in a sturctured JSON list.\n\n"
        formatted_input += f"Based on the following prescribing information, provide the drug-drug interactions for the drug, {ex['drug']}, in a sturctured JSON list.\n\n"
        output_input += "TEXT OMITTED"
        formatted_input += ex['text']

        output_input += "<end_of_turn>\n<start_of_turn>model\n"
        formatted_input += "<end_of_turn>\n<start_of_turn>model\n"
        output_input += json.dumps(ex['interactions']) + "<end_of_turn>\n"
        formatted_input += json.dumps(ex['interactions']) + "<end_of_turn>\n"

    # Adds the actual question/prompt
    output_input += f"<start_of_turn>user\nBased on the following prescribing information, provide the drug-drug interactions for the drug, {target_drug}, in a sturctured JSON list.\n\n"
    formatted_input += f"<start_of_turn>user\nBased on the following prescribing information, provide the drug-drug interactions for the drug, {target_drug}, in a sturctured JSON list.\n\n"
    output_input += "TEXT OMITTED"
    formatted_input += target_text
    output_input += "<end_of_turn>\n<start_of_turn>model\n[" # Force JSON start
    formatted_input += "<end_of_turn>\n<start_of_turn>model\n[" # Force JSON start

    # Visual
    display(Markdown("---"))
    display(Markdown(f"**The following text is being sent to the model:**\n\n```text\n{output_input}\n```"))

    # Execute
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.01,
            do_sample=False
        )

    # Extract and format response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    final_json = "[" + response

    display(Markdown(f"**MODEL RESPONSE:**\n\n```json\n{final_json}\n```"))
    return final_json

## Sample it

In [7]:
# INPUT YOUR TEST CASE HERE
my_test_drug = "Crestor" # @param {type: "string"}
my_test_text = "Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K)." # @param {type: "string"}

# Execute
result = run_in_context_extraction(my_test_drug, my_test_text)

---

**The following text is being sent to the model:**

```text
<start_of_turn>user
You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

Based on the following prescribing information, provide the drug-drug interactions for the drug, Xarelto, in a sturctured JSON list.

WARNING: (A) PREMATURE DISCONTINUATION OF XARELTO INCREASES THE RISK OF THROMBOTIC EVENTS, (B) SPINAL/EPIDURAL HEMATOMA 

 WARNING: (A) PREMATURE DISCONTINUATION OF XARELTO INCREASES THE RISK OF THROMBOTIC EVENTS, (B) SPINAL/EPIDURAL HEMATOMA 

 See full prescribing information for complete boxed warning 

 (A) Premature discontinuation of XARELTO increases the risk of thrombotic events 

 Premature discontinuation of any oral anticoagulant, including XARELTO, increases the risk of thrombotic events. To reduce this risk, consider coverage with another anticoagulant if XARELTO is discontinued for a reason other than pathological bleeding or completion of a course of therapy ( 2.3 , 2.8 , 5.1 , 14.1 ). 

 (B) Spinal/epidural hematoma 

 Epidural or spinal hematomas have occurred in patients treated with XARELTO who are receiving neuraxial anesthesia or undergoing spinal puncture. These hematomas may result in long-term or permanent paralysis ( 5.2 , 5.3 , 6.2 ). 

 Monitor patients frequently for signs and symptoms of neurological impairment and if observed, treat urgently. Consider the benefits and risks before neuraxial intervention in patients who are or who need to be anticoagulated ( 5.3 ). 

 A. Premature discontinuation of XARELTO increases the risk of thrombotic events 

 Premature discontinuation of any oral anticoagulant, including XARELTO, increases the risk of thrombotic events. If anticoagulation with XARELTO is discontinued for a reason other than pathological bleeding or completion of a course of therapy, consider coverage with another anticoagulant [see Dosage and Administration (2.3 , 2.8) , Warnings and Precautions (5.1) , and Clinical Studies (14.1) ] . 

 B. Spinal/epidural hematoma 

 Epidural or spinal hematomas have occurred in patients treated with XARELTO who are receiving neuraxial anesthesia or undergoing spinal puncture. These hematomas may result in long-term or permanent paralysis. Consider these risks when scheduling patients for spinal procedures. Factors that can increase the risk of developing epidural or spinal hematomas in these patients include: 

 use of indwelling epidural catheters 

 concomitant use of other drugs that affect hemostasis, such as non-steroidal anti-inflammatory drugs (NSAIDs), platelet inhibitors, other anticoagulants 

 a history of traumatic or repeated epidural or spinal punctures 

 a history of spinal deformity or spinal surgery 

 optimal timing between the administration of XARELTO and neuraxial procedures is not known 

 [see Warnings and Precautions (5.2 , 5.3) and Adverse Reactions (6.2) ]. 

 Monitor patients frequently for signs and symptoms of neurological impairment. If neurological compromise is noted, urgent treatment is necessary [see Warnings and Precautions (5.3) ] . 

 Consider the benefits and risks before neuraxial intervention in patients anticoagulated or to be anticoagulated for thromboprophylaxis [see Warnings and Precautions (5.3) ] .2 DOSAGE AND ADMINISTRATION 

 Nonvalvular Atrial Fibrillation : 
 For patients with CrCl >50 mL/min: 20 mg orally, once daily with the evening meal ( 2.4 ) 
 For patients with CrCl 15 – 50 mL/min: 15 mg orally, once daily with the evening meal ( 2.4 ) 

 Treatment of DVT and/or PE : 15 mg orally twice daily with food for the first 21 days followed by 20 mg orally once daily with food for the remaining treatment ( 2.5 ) 

 Reduction in the Risk of Recurrence of DVT and/or PE in patients at continued risk for DVT and/or PE : 10 mg once daily with or without food, after at least 6 months of standard anticoagulant treatment ( 2.6 ) 

 Prophylaxis of DVT Following Hip or Knee Replacement Surgery : 10 mg orally once daily with or without food ( 2.7 ) 

 2.1 Recommended Dosage 

 Indication 
 Dosage 

 Reduction in Risk of Stroke in Nonvalvular Atrial Fibrillation (2.4) 

 CrCl >50 mL/min: 
 20 mg once daily with the evening meal 

 CrCl 15 to 50 mL/min: 
 15 mg once daily with the evening meal 

 Treatment of DVT (2.5) Treatment of PE (2.5) 

 15 mg twice daily with food, for first 21 days 

     ▼after 21 days, transition to ▼ 

 20 mg once daily with food, for remaining treatment 

 Reduction in the Risk of Recurrence of DVT and/or PE in patients at continued risk for DVT and/or PE (2.6) 

 10 mg once daily with or without food, after at least 6 months of standard anticoagulant treatment 

 Prophylaxis of DVT Following Hip or Knee Replacement Surgery (2.7) 

 Hip replacement: 
 10 mg once daily with or without food for 35 days 

 Knee replacement: 
 10 mg once daily with or without food for 12 days 

 2.2 Important Food Effect Information 

 The 15 mg and 20 mg XARELTO tablets should be taken with food, while the 10 mg tablet can be taken with or without food [see Clinical Pharmacology (12.3) ] . 
 In the nonvalvular atrial fibrillation efficacy study XARELTO was taken with the evening meal. 

 2.3 Switching to and from XARELTO 

 Switching from Warfarin to XARELTO - When switching patients from warfarin to XARELTO, discontinue warfarin and start XARELTO as soon as the International Normalized Ratio (INR) is below 3.0 to avoid periods of inadequate anticoagulation. 

 Switching from XARELTO to Warfarin - No clinical trial data are available to guide converting patients from XARELTO to warfarin. XARELTO affects INR, so INR measurements made during coadministration with warfarin may not be useful for determining the appropriate dose of warfarin. One approach is to discontinue XARELTO and begin both a parenteral anticoagulant and warfarin at the time the next dose of XARELTO would have been taken. 

 Switching from XARELTO to Anticoagulants other than Warfarin - For patients currently taking XARELTO and transitioning to an anticoagulant with rapid onset, discontinue XARELTO and give the first dose of the other anticoagulant (oral or parenteral) at the time that the next XARELTO dose would have been taken [see Drug Interactions (7.4) ] . 

 Switching from Anticoagulants other than Warfarin to XARELTO - For patients currently receiving an anticoagulant other than warfarin, start XARELTO 0 to 2 hours prior to the next scheduled evening administration of the drug (e.g., low molecular weight heparin or non-warfarin oral anticoagulant) and omit administration of the other anticoagulant. For unfractionated heparin being administered by continuous infusion, stop the infusion and start XARELTO at the same time. 

 2.4 Nonvalvular Atrial Fibrillation 

 For patients with creatinine clearance (CrCl) >50 mL/min, the recommended dose of XARELTO is 20 mg taken orally once daily with the evening meal. For patients with CrCl 15 to 50 mL/min, the recommended dose is 15 mg once daily with the evening meal [see Use in Specific Populations (8.6) ] . 

 2.5 Treatment of Deep Vein Thrombosis (DVT) and/or Pulmonary Embolism (PE) 

 The recommended dose of XARELTO for the initial treatment of acute DVT and/or PE is 15 mg taken orally twice daily with food for the first 21 days. After this initial treatment period, the recommended dose of XARELTO is 20 mg taken orally once daily with food, at approximately the same time each day [see Clinical Studies (14.2) ] . 

 2.6 Reduction in the Risk of Recurrence of Deep Vein Thrombosis (DVT) and/or Pulmonary Embolism (PE) 

 The recommended dose of XARELTO for the reduction in the risk of recurrence of DVT and/or PE after at least 6 months of standard anticoagulant treatment in patients at continued risk of DVT and/or PE is 10 mg taken orally once daily with or without food [see Clinical Studies (14.3) ]. 

 2.7 Prophylaxis of Deep Vein Thrombosis Following Hip or Knee Replacement Surgery 

 The recommended dose of XARELTO is 10 mg taken orally once daily with or without food. The initial dose should be taken 6 to 10 hours after surgery provided that hemostasis has been established [see Dosage and Administration (2.8) ] . 

 For patients undergoing hip replacement surgery, treatment duration of 35 days is recommended. 
 For patients undergoing knee replacement surgery, treatment duration of 12 days is recommended. 

 2.8 Discontinuation for Surgery and other Interventions 

 If anticoagulation must be discontinued to reduce the risk of bleeding with surgical or other procedures, XARELTO should be stopped at least 24 hours before the procedure to reduce the risk of bleeding [see Warnings and Precautions (5.2) ] . In deciding whether a procedure should be delayed until 24 hours after the last dose of XARELTO, the increased risk of bleeding should be weighed against the urgency of intervention. XARELTO should be restarted after the surgical or other procedures as soon as adequate hemostasis has been established, noting that the time to onset of therapeutic effect is short [see Warnings and Precautions (5.1) ] . If oral medication cannot be taken during or after surgical intervention, consider administering a parenteral anticoagulant. 

 2.9 Missed Dose 

 If a dose of XARELTO is not taken at the scheduled time, administer the dose as soon as possible on the same day as follows: 

 For patients receiving 15 mg twice daily: The patient should take XARELTO immediately to ensure intake of 30 mg XARELTO per day. In this particular instance, two 15 mg tablets may be taken at once. The patient should continue with the regular 15 mg twice daily intake as recommended on the following day. 
 For patients receiving 20 mg, 15 mg or 10 mg once daily: The patient should take the missed XARELTO dose immediately. 

 2.10 Administration Options 

 For patients who are unable to swallow whole tablets, 10 mg, 15 mg or 20 mg XARELTO tablets may be crushed and mixed with applesauce immediately prior to use and administered orally. After the administration of a crushed XARELTO 15 mg or 20 mg tablet, the dose should be immediately followed by food [see Dosage and Administration (2.2 , 2.4 , 2.5) and Clinical Pharmacology (12.3) ] . 

 Administration via nasogastric (NG) tube or gastric feeding tube: After confirming gastric placement of the tube, 10 mg, 15 mg or 20 mg XARELTO tablets may be crushed and suspended in 50 mL of water and administered via an NG tube or gastric feeding tube. Since rivaroxaban absorption is dependent on the site of drug release, avoid administration of XARELTO distal to the stomach which can result in reduced absorption and thereby, reduced drug exposure. After the administration of a crushed XARELTO 15 mg or 20 mg tablet, the dose should then be immediately followed by enteral feeding [see Clinical Pharmacology (12.3) ] . 
 Crushed 10 mg, 15 mg or 20 mg XARELTO tablets are stable in water and in applesauce for up to 4 hours. An in vitro compatibility study indicated that there is no adsorption of rivaroxaban from a water suspension of a crushed XARELTO tablet to PVC or silicone nasogastric (NG) tubing.4 CONTRAINDICATIONS 

 XARELTO is contraindicated in patients with: 

 active pathological bleeding [see Warnings and Precautions (5.2) ] 

 severe hypersensitivity reaction to XARELTO (e.g., anaphylactic reactions) [see Adverse Reactions (6.2) ] 

 Active pathological bleeding ( 4 ) 
 Severe hypersensitivity reaction to XARELTO ( 4 )5 WARNINGS AND PRECAUTIONS 

 Risk of bleeding: XARELTO can cause serious and fatal bleeding. Promptly evaluate signs and symptoms of blood loss. ( 5.2 ) 
 Pregnancy-related hemorrhage: Use XARELTO with caution in pregnant women due to the potential for obstetric hemorrhage and/or emergent delivery. Promptly evaluate signs and symptoms of blood loss. ( 5.7 ) 
 Prosthetic heart valves: XARELTO use not recommended ( 5.8 ) 

 5.1 Increased Risk of Thrombotic Events after Premature Discontinuation 

 Premature discontinuation of any oral anticoagulant, including XARELTO, in the absence of adequate alternative anticoagulation increases the risk of thrombotic events. An increased rate of stroke was observed during the transition from XARELTO to warfarin in clinical trials in atrial fibrillation patients. If XARELTO is discontinued for a reason other than pathological bleeding or completion of a course of therapy, consider coverage with another anticoagulant [see Dosage and Administration (2.3 , 2.8) and Clinical Studies (14.1) ] . 

 5.2 Risk of Bleeding 

 XARELTO increases the risk of bleeding and can cause serious or fatal bleeding. In deciding whether to prescribe XARELTO to patients at increased risk of bleeding, the risk of thrombotic events should be weighed against the risk of bleeding. 
 Promptly evaluate any signs or symptoms of blood loss and consider the need for blood replacement. Discontinue XARELTO in patients with active pathological hemorrhage. The terminal elimination half-life of rivaroxaban is 5 to 9 hours in healthy subjects aged 20 to 45 years. 
 Concomitant use of other drugs that impair hemostasis increases the risk of bleeding. These include aspirin, P2Y 12 platelet inhibitors, other antithrombotic agents, fibrinolytic therapy, non-steroidal anti-inflammatory drugs (NSAIDs) [see Drug Interactions (7.4) ] , selective serotonin reuptake inhibitors, and serotonin norepinephrine reuptake inhibitors. 
 Concomitant use of drugs that are known combined P-gp and strong CYP3A4 inhibitors increases rivaroxaban exposure and may increase bleeding risk [see Drug Interactions (7.2) ] . 

 Reversal of Anticoagulant Effect 

 A specific antidote for rivaroxaban is not available. Because of high plasma protein binding, rivaroxaban is not expected to be dialyzable [see Clinical Pharmacology (12.3) ] . Protamine sulfate and vitamin K are not expected to affect the anticoagulant activity of rivaroxaban. Partial reversal of prothrombin time prolongation has been seen after administration of prothrombin complex concentrates (PCCs) in healthy volunteers. The use of other procoagulant reversal agents like activated prothrombin complex concentrate (APCC) or recombinant factor VIIa (rFVIIa) has not been evaluated. 

 5.3 Spinal/Epidural Anesthesia or Puncture 

 When neuraxial anesthesia (spinal/epidural anesthesia) or spinal puncture is employed, patients treated with anticoagulant agents for prevention of thromboembolic complications are at risk of developing an epidural or spinal hematoma which can result in long-term or permanent paralysis [see Boxed Warning ] . 
 To reduce the potential risk of bleeding associated with the concurrent use of XARELTO and epidural or spinal anesthesia/analgesia or spinal puncture, consider the pharmacokinetic profile of XARELTO [see Clinical Pharmacology (12.3) ] . Placement or removal of an epidural catheter or lumbar puncture is best performed when the anticoagulant effect of XARELTO is low; however, the exact timing to reach a sufficiently low anticoagulant effect in each patient is not known. 
 An indwelling epidural or intrathecal catheter should not be removed before at least 2 half-lives have elapsed (i.e., 18 hours in young patients aged 20 to 45 years and 26 hours in elderly patients aged 60 to 76 years), after the last administration of XARELTO [see Clinical Pharmacology (12.3) ] . The next XARELTO dose should not be administered earlier than 6 hours after the removal of the catheter. If traumatic puncture occurs, delay the administration of XARELTO for 24 hours. 
 Should the physician decide to administer anticoagulation in the context of epidural or spinal anesthesia/analgesia or lumbar puncture, monitor frequently to detect any signs or symptoms of neurological impairment, such as midline back pain, sensory and motor deficits (numbness, tingling, or weakness in lower limbs), bowel and/or bladder dysfunction. Instruct patients to immediately report if they experience any of the above signs or symptoms. If signs or symptoms of spinal hematoma are suspected, initiate urgent diagnosis and treatment including consideration for spinal cord decompression even though such treatment may not prevent or reverse neurological sequelae. 

 5.4 Use in Patients with Renal Impairment 

 Nonvalvular Atrial Fibrillation 

 Periodically assess renal function as clinically indicated (i.e., more frequently in situations in which renal function may decline) and adjust therapy accordingly [see Dosage and Administration (2.4) ] . Consider dose adjustment or discontinuation of XARELTO in patients who develop acute renal failure while on XARELTO [see Use in Specific Populations (8.6) ]. 

 Treatment of Deep Vein Thrombosis (DVT), Pulmonary Embolism (PE), and Reduction in the Risk of Recurrence of DVT and of PE 

 Avoid the use of XARELTO in patients with CrCl <30 mL/min due to an expected increase in rivaroxaban exposure and pharmacodynamic effects in this patient population [see Use in Specific Populations (8.6) ] . 

 Prophylaxis of Deep Vein Thrombosis Following Hip or Knee Replacement Surgery 

 Avoid the use of XARELTO in patients with CrCl <30 mL/min due to an expected increase in rivaroxaban exposure and pharmacodynamic effects in this patient population. Observe closely and promptly evaluate any signs or symptoms of blood loss in patients with CrCl 30 to 50 mL/min. Patients who develop acute renal failure while on XARELTO should discontinue the treatment [see Use in Specific Populations (8.6) ]. 

 5.5 Use in Patients with Hepatic Impairment 

 No clinical data are available for patients with severe hepatic impairment. 
 Avoid use of XARELTO in patients with moderate (Child-Pugh B) and severe (Child-Pugh C) hepatic impairment or with any hepatic disease associated with coagulopathy since drug exposure and bleeding risk may be increased [see Use in Specific Populations (8.7) ] . 

 5.6 Use with P-gp and Strong CYP3A4 Inhibitors or Inducers 

 Avoid concomitant use of XARELTO with known combined P-gp and strong CYP3A4 inhibitors [see Drug Interactions (7.2) ] . 
 Avoid concomitant use of XARELTO with drugs that are known combined P-gp and strong CYP3A4 inducers [see Drug Interactions (7.3) ] . 

 5.7 Risk of Pregnancy-Related Hemorrhage 

 In pregnant women, XARELTO should be used only if the potential benefit justifies the potential risk to the mother and fetus. XARELTO dosing in pregnancy has not been studied. The anticoagulant effect of XARELTO cannot be monitored with standard laboratory testing nor readily reversed. Promptly evaluate any signs or symptoms suggesting blood loss (e.g., a drop in hemoglobin and/or hematocrit, hypotension, or fetal distress). 

 5.8 Patients with Prosthetic Heart Valves 

 The safety and efficacy of XARELTO have not been studied in patients with prosthetic heart valves. Therefore, use of XARELTO is not recommended in these patients. 

 5.9 Acute PE in Hemodynamically Unstable Patients or Patients Who Require Thrombolysis or Pulmonary Embolectomy 

 Initiation of XARELTO is not recommended acutely as an alternative to unfractionated heparin in patients with pulmonary embolism who present with hemodynamic instability or who may receive thrombolysis or pulmonary embolectomy.7 DRUG INTERACTIONS 

 Combined P-gp and strong CYP3A4 inhibitors and inducers: Avoid concomitant use ( 7.2 , 7.3 ) 
 Anticoagulants: Avoid concomitant use ( 7.4 ) 

 7.1 General Inhibition and Induction Properties 

 Rivaroxaban is a substrate of CYP3A4/5, CYP2J2, and the P-gp and ATP-binding cassette G2 (ABCG2) transporters. Combined P-gp and strong CYP3A4 inhibitors increase exposure to rivaroxaban and may increase the risk of bleeding. Combined P-gp and strong CYP3A4 inducers decrease exposure to rivaroxaban and may increase the risk of thromboembolic events. 

 7.2 Drugs that Inhibit Cytochrome P450 3A4 Enzymes and Drug Transport Systems 

 Interaction with Combined P-gp and Strong CYP3A4 Inhibitors 

 Avoid concomitant administration of XARELTO with known combined P-gp and strong CYP3A4 inhibitors (e.g., ketoconazole and ritonavir) [see Warnings and Precautions (5.6) and Clinical Pharmacology (12.3) ] . 
 Although clarithromycin is a combined P-gp and strong CYP3A4 inhibitor, pharmacokinetic data suggests that no precautions are necessary with concomitant administration with XARELTO as the change in exposure is unlikely to affect the bleeding risk [see Clinical Pharmacology (12.3) ] . 

 Interaction with Combined P-gp and Moderate CYP3A4 Inhibitors in Patients with Renal Impairment 

 XARELTO should not be used in patients with CrCl 15 to <80 mL/min who are receiving concomitant combined P-gp and moderate CYP3A4 inhibitors (e.g., erythromycin) unless the potential benefit justifies the potential risk [see Warnings and Precautions (5.4) and Clinical Pharmacology (12.3) ] . 

 7.3 Drugs that Induce Cytochrome P450 3A4 Enzymes and Drug Transport Systems 

 Avoid concomitant use of XARELTO with drugs that are combined P-gp and strong CYP3A4 inducers (e.g., carbamazepine, phenytoin, rifampin, St. John's wort) [see Warnings and Precautions (5.6) and Clinical Pharmacology (12.3) ] . 

 7.4 Anticoagulants and NSAIDs/Aspirin 

 Coadministration of enoxaparin, warfarin, aspirin, clopidogrel and chronic NSAID use may increase the risk of bleeding [see Clinical Pharmacology (12.3) ]. 

 Avoid concurrent use of XARELTO with other anticoagulants due to increased bleeding risk unless benefit outweighs risk. Promptly evaluate any signs or symptoms of blood loss if patients are treated concomitantly with aspirin, other platelet aggregation inhibitors, or NSAIDs [see Warnings and Precautions (5.2) ] .<end_of_turn>
<start_of_turn>model
[{"type": "Unspecified interaction", "precipitant": "ketoconazole", "precipitantCode": "N0000007319"}, {"type": "Pharmacodynamic interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacokinetic interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503", "effect": "C54355"}, {"type": "Unspecified interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503"}, {"type": "Unspecified interaction", "precipitant": "ritonavir", "precipitantCode": "N0000007423"}, {"type": "Pharmacodynamic interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacokinetic interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141", "effect": "C54355"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141"}, {"type": "Unspecified interaction", "precipitant": "carbamazepine", "precipitantCode": "N0000007470"}, {"type": "Pharmacodynamic interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264", "effect": "371039008: Thromboembolic disorder (disorder)"}, {"type": "Pharmacokinetic interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264", "effect": "C54356"}, {"type": "Unspecified interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264"}, {"type": "Unspecified interaction", "precipitant": "phenytoin", "precipitantCode": "N0000006023"}, {"type": "Unspecified interaction", "precipitant": "rifampin", "precipitantCode": "N0000006026"}, {"type": "Unspecified interaction", "precipitant": "st.   john 's   wort", "precipitantCode": "N0000022261"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 inducers", "precipitantCode": "n0000185506"}, {"type": "Pharmacodynamic interaction", "precipitant": "anticoagulants", "precipitantCode": "N0000029110", "effect": "131148009: Bleeding (finding)"}, {"type": "Unspecified interaction", "precipitant": "anticoagulants", "precipitantCode": "N0000029110"}, {"type": "Pharmacodynamic interaction", "precipitant": "aspirin", "precipitantCode": "N0000006582", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "chronic nsaid", "precipitantCode": "NO MAP", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "clopidogrel", "precipitantCode": "N0000007277", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "enoxaparin", "precipitantCode": "N0000006809", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "warfarin", "precipitantCode": "N0000006403", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "cyp3a4 inducers", "precipitantCode": "N0000185506", "effect": "371039008: Thromboembolic disorder (disorder)"}, {"type": "Pharmacokinetic interaction", "precipitant": "cyp3a4 inducers", "precipitantCode": "N0000185506", "effect": "C54356"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 | inducers", "precipitantCode": "n0000185506"}, {"type": "Unspecified interaction", "precipitant": "erythromycin", "precipitantCode": "N0000007133"}, {"type": "Unspecified interaction", "precipitant": "moderate cyp3a4 inhibitors", "precipitantCode": "n0000182141"}, {"type": "Pharmacodynamic interaction", "precipitant": "nsaids", "precipitantCode": "N0000175722", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "platelet aggregation inhibitors", "precipitantCode": "N0000029299", "effect": "131148009: Bleeding (finding)"}]<end_of_turn>
<start_of_turn>user
Based on the following prescribing information, provide the drug-drug interactions for the drug, ADCIRCA, in a sturctured JSON list.

2 DOSAGE AND ADMINISTRATION

 40 mg once daily, with or without food. ( 2.1 )

 Dividing the dose (40 mg) over the course of the day is not recommended. ( 2.1 )

 Use with ritonavir requires dosage adjustments. ( 2.3 )

 2.1 Pulmonary Arterial Hypertension

 The recommended dose of ADCIRCA is 40 mg (two 20 mg tablets) taken once daily with or without food. Dividing the dose (40 mg) over the course of the day is not recommended.

 2.2 Use in Special Populations

 Renal Impairment 

 Mild (creatinine clearance 51 to 80 mL/min) or moderate (creatinine clearance 31 to 50 mL/min): Start dosing at 20 mg once daily. Increase to 40 mg once daily based on individual tolerability.

 Severe (creatinine clearance <30 mL/min and on hemodialysis): Avoid use of ADCIRCA because of increased tadalafil exposure (AUC), limited clinical experience, and the lack of ability to influence clearance by dialysis [see Warnings and Precautions ( 5.3 ) and Use in Specific Populations ( 8.6 )] .

 Hepatic Impairment 

 Mild or moderate (Child Pugh Class A or B): Because of limited clinical experience in patients with mild to moderate hepatic cirrhosis, consider a starting dose of 20 mg once per day.

 Severe (Child Pugh Class C): Patients with severe hepatic cirrhosis have not been studied. Avoid use of ADCIRCA [see Warnings and Precautions ( 5.4 ) and Use in Specific Populations ( 8.7 )] .

 Geriatric Patients 

 No dose adjustment is required in patients >65 years of age without renal impairment or hepatic impairment.

 2.3 Use with Ritonavir

 Co-administration of ADCIRCA in Patients on Ritonavir 

 In patients receiving ritonavir for at least one week, start ADCIRCA at 20 mg once daily. Increase to 40 mg once daily based upon individual tolerability [see Warnings and Precautions ( 5.2 ), Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .

 Co-administration of Ritonavir in Patients on ADCIRCA 

 Avoid use of ADCIRCA during the initiation of ritonavir. Stop ADCIRCA at least 24 hours prior to starting ritonavir. After at least one week following the initiation of ritonavir, resume ADCIRCA at 20 mg once daily. Increase to 40 mg once daily based upon individual tolerability [see Warnings and Precautions ( 5.2 ), Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .4 CONTRAINDICATIONS

 Concomitant organic nitrates ( 4.1 )

 Concomitant Guanylate Cyclase (GC) Stimulators ( 4.2 ) 
 History of known serious hypersensitivity reaction to ADCIRCA or CIALIS ( 4.3 )

 4.1 Concomitant Organic Nitrates

 Do not use ADCIRCA in patients who are using any form of organic nitrate, either regularly or intermittently. ADCIRCA potentiates the hypotensive effect of nitrates. This potentiation is thought to result from the combined effects of nitrates and ADCIRCA on the nitric oxide/cGMP pathway [see Clinical Pharmacology ( 12.2 )] .

 4.2 Concomitant Guanylate Cyclase (GC) Stimulators

 Do not use ADCIRCA in patients who are using a GC stimulator, such as riociguat. ADCIRCA may potentiate the hypotensive effects of GC stimulators.

 4.3 Hypersensitivity Reactions

 ADCIRCA is contraindicated in patients with a known serious hypersensitivity to tadalafil (ADCIRCA or CIALIS). Hypersensitivity reactions have been reported, including Stevens-Johnson syndrome and exfoliative dermatitis [see Adverse Reactions ( 6.2 )] .5 WARNINGS AND PRECAUTIONS

 Cardiovascular effects: Carefully consider whether patients with certain underlying conditions (e.g., cardiovascular disease, impaired autonomic control of blood pressure, aortic stenosis) could be adversely affected by vasodilatory effects of ADCIRCA. Not recommended in patients with pulmonary veno-occlusive disease. ( 5.1 )

 Concomitant alpha-blockers or alcohol: Note additive blood pressure-lowering effects. ( 5.1 )

 Use with Ritonavir: Requires dosage adjustment. ( 2.3 , 5.2 ) 
 Other concomitant potent CYP3A inhibitors: Avoid use with ADCIRCA. ( 5.2 )

 Potent Inducers of CYP3A: Avoid use of ADCIRCA in patients chronically taking potent inducers of CYP3A (e.g., rifampin). ( 5.2 , 7.2 )

 Effects on the eye: Patients should seek immediate medical attention if sudden loss of vision occurs, which could be a sign of non-arteritic ischemic optic neuropathy (NAION). ( 5.5 )

 Hearing impairment: Advise patients to seek immediate medical attention if sudden decrease or loss of hearing occurs. ( 5.6 )

 Concomitant PDE5 inhibitors: Avoid use with CIALIS or other PDE5 inhibitors. ( 5.7 )

 Prolonged erection: Advise patients to seek emergency treatment if an erection lasts >4 hours. ( 5.8 )

 5.1 Cardiovascular Effects

 Discuss with patients the appropriate action to take in the event that they experience anginal chest pain requiring nitroglycerin following intake of ADCIRCA. At least 48 hours should elapse after the last dose of ADCIRCA before taking nitrates. If a patient has taken ADCIRCA within 48 hours, administer nitrates under close medical supervision with appropriate hemodynamic monitoring. Patients who experience anginal chest pain after taking ADCIRCA should seek immediate medical attention.

 PDE5 inhibitors, including tadalafil, have mild systemic vasodilatory properties that may result in transient decreases in blood pressure. Prior to prescribing ADCIRCA, carefully consider whether patients with underlying cardiovascular disease could be affected adversely by such vasodilatory effects. Patients with severely impaired autonomic control of blood pressure or with left ventricular outflow obstruction, (e.g., aortic stenosis and idiopathic hypertrophic subaortic stenosis) may be particularly sensitive to the actions of vasodilators, including PDE5 inhibitors.

 Pulmonary vasodilators may significantly worsen the cardiovascular status of patients with pulmonary veno-occlusive disease (PVOD). Since there are no clinical data on administration of ADCIRCA to patients with veno-occlusive disease, administration of ADCIRCA to such patients is not recommended. Should signs of pulmonary edema occur when ADCIRCA is administered, the possibility of associated PVOD should be considered.

 There is a lack of data on safety and efficacy in the following groups who were specifically excluded from the PAH clinical trials:

 Patients with clinically significant aortic and mitral valve disease

 Patients with pericardial constriction

 Patients with restrictive or congestive cardiomyopathy

 Patients with significant left ventricular dysfunction

 Patients with life-threatening arrhythmias

 Patients with symptomatic coronary artery disease

 Patients with hypotension (<90/50 mm Hg) or uncontrolled hypertension

 Use with Alpha Blockers and Antihypertensives 

 PDE5 inhibitors, including ADCIRCA, and alpha–adrenergic blocking agents are vasodilators with blood pressure–lowering effects. When vasodilators are used in combination, an additive effect on blood pressure may be anticipated. In some patients, concomitant use of these two drug classes can lower blood pressure significantly [see Drug Interactions ( 7.1 ) and Clinical Pharmacology ( 12.2 )] , which may lead to symptomatic hypotension (e.g., fainting). Safety of combined use of PDE5 inhibitors and alpha blockers may be affected by other variables, including intravascular volume depletion and use of other antihypertensive drugs [see Drug Interactions ( 7.1 )] .

 Use with Alcohol 

 Both alcohol and tadalafil are mild vasodilators. When mild vasodilators are taken in combination, blood pressure-lowering effects are increased [see Drug Interactions ( 7.1 ) and Clinical Pharmacology ( 12.2 )] .

 5.2 Use with Potent CYP3A Inhibitors or Inducers

 Co-administration of ADCIRCA in Patients on Ritonavir 

 In patients receiving ritonavir for at least one week, start ADCIRCA at 20 mg once daily. Increase to 40 mg once daily based upon individual tolerability [see Dosage and Administration ( 2.3 ), Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .

 Co-administration of Ritonavir in Patients on ADCIRCA 

 Avoid use of ADCIRCA during the initiation of ritonavir. Stop ADCIRCA at least 24 hours prior to starting ritonavir. After at least one week following the initiation of ritonavir, resume ADCIRCA at 20 mg once daily. Increase to 40 mg once daily based upon individual tolerability [see Dosage and Administration ( 2.3 ), Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .

 Other Potent Inhibitors of CYP3A 

 Tadalafil is metabolized predominantly by CYP3A in the liver. In patients taking potent inhibitors of CYP3A such as ketoconazole and itraconazole, avoid use of ADCIRCA [see Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .

 Potent Inducers of CYP3A 

 For patients chronically taking potent inducers of CYP3A, such as rifampin, avoid use of ADCIRCA [see Drug Interactions ( 7.2 ) and Clinical Pharmacology ( 12.3 )] .

 5.3 Use in Renal Impairment

 In patients with mild or moderate renal impairment 

 Start dosing at 20 mg once daily. Increase the dose to 40 mg once daily based upon individual tolerability [see Dosage and Administration ( 2.2 ) and Clinical Pharmacology ( 12.3 )] .

 In patients with severe renal impairment 

 Avoid use of ADCIRCA because of increased tadalafil exposure (AUC), limited clinical experience, and the lack of ability to influence clearance by dialysis [see Dosage and Administration ( 2.2 ) and Clinical Pharmacology ( 12.3 )] .

 5.4 Use in Hepatic Impairment

 In patients with mild to moderate hepatic cirrhosis (Child-Pugh Class A and B) 

 Because of limited clinical experience in patients with mild to moderate hepatic cirrhosis, consider a starting dose of 20 mg once daily ADCIRCA [see Dosage and Administration ( 2.2 ) and Clinical Pharmacology ( 12.3 )] .

 In patients with severe hepatic cirrhosis (Child-Pugh Class C) 

 Patients with severe hepatic cirrhosis have not been studied. Avoid use of ADCIRCA [see Dosage and Administration ( 2.2 ) and Clinical Pharmacology ( 12.3 )] .

 5.5 Visual Loss

 Physicians should advise patients to seek immediate medical attention in the event of a sudden loss of vision in one or both eyes. Such an event may be a sign of non–arteritic anterior ischemic optic neuropathy (NAION), a cause of decreased vision, including permanent loss of vision, that has been reported postmarketing in temporal association with the use of all PDE5 inhibitors. Most, but not all, of these patients had underlying anatomic or vascular risk factors for development of NAION, including but not necessarily limited to: low cup to disc ratio (“crowded disc”), age over 50, diabetes, hypertension, coronary artery disease, hyperlipidemia, and smoking. Based on published literature, the annual incidence of NAION is 2.5-11.8 cases per 100,000 in males aged ≥50 in the general population. An observational case-crossover study evaluated the risk of NAION when PDE5 inhibitor use, as a class, typical of erectile dysfunction treatment, occurred immediately before NAION onset (within 5 half-lives), compared to PDE5 inhibitor use in a prior time period. The results suggest an approximate 2-fold increase in the risk of NAION, with a risk estimate of 2.15 (95% CI 1.06, 4.34). A similar study reported a consistent result, with a risk estimate of 2.27 (95% CI 0.99, 5.20). Other risk factors for NAION, such as the presence of “crowded” optic disc, may have contributed to the occurrence of NAION in these studies.

 Neither the rare postmarketing reports, nor the association of PDE5 inhibitor use and NAION in the observational studies, substantiate a causal relationship between PDE5 inhibitor use and NAION [see Adverse Reactions ( 6.2 )] . 
 Physicians should also discuss with patients the increased risk of NAION in individuals who have already experienced NAION in one eye, including whether such individuals could be adversely affected by use of vasodilators such as PDE5 inhibitors. 
 Patients with known hereditary degenerative retinal disorders, including retinitis pigmentosa, were not included in the clinical trials, and use in these patients is not recommended.

 5.6 Hearing Impairment

 Physicians should advise patients to seek immediate medical attention in the event of sudden decrease or loss of hearing. These events, which may be accompanied by tinnitus and dizziness, have been reported in temporal association to the intake of PDE5 inhibitors, including ADCIRCA. It is not possible to determine whether these events are related directly to the use of PDE5 inhibitors or to other factors [see Adverse Reactions ( 6.2 )] .

 5.7 Combination with Other PDE5 Inhibitors

 Tadalafil is also marketed as CIALIS. The safety and efficacy of taking ADCIRCA together with CIALIS or other PDE5 inhibitors have not been studied. Inform patients taking ADCIRCA not to take CIALIS or other PDE5 inhibitors.

 5.8 Prolonged Erection

 There have been rare reports of prolonged erections greater than 4 hours and priapism (painful erections greater than 6 hours in duration) for this class of compounds. Priapism, if not treated promptly, can result in irreversible damage to the erectile tissue. Patients who have an erection lasting greater than 4 hours, whether painful or not, should seek emergency medical attention.

 ADCIRCA should be used with caution in patients who have conditions that might predispose them to priapism (such as sickle cell anemia, multiple myeloma, or leukemia), or in patients with anatomical deformation of the penis (such as angulation, cavernosal fibrosis, or Peyronie's disease).

 5.9 Effects on Bleeding

 PDE5 is found in platelets. When administered in combination with aspirin, tadalafil 20 mg did not prolong bleeding time, relative to aspirin alone. ADCIRCA has not been administered to patients with bleeding disorders or significant active peptic ulceration. Although ADCIRCA has not been shown to increase bleeding times in healthy subjects, use in patients with bleeding disorders or significant active peptic ulceration should be based upon a careful risk-benefit assessment.7 DRUG INTERACTIONS

 7.1 Potential for Pharmacodynamic Interactions with ADCIRCA

 Nitrates 

 Do not use ADCIRCA in patients who are using any form of organic nitrate [see Contraindications ( 4.1 )] . In clinical pharmacology studies ADCIRCA potentiated the hypotensive effect of nitrates [see Clinical Pharmacology ( 12.2 )] . In a patient who has taken ADCIRCA, where nitrate administration is deemed medically necessary in a life–threatening situation, at least 48 hours should elapse after the last dose of ADCIRCA before nitrate administration is considered. In such circumstances, nitrates should still only be administered under close medical supervision with appropriate hemodynamic monitoring.

 Alpha-Blockers 

 PDE5 inhibitors, including ADCIRCA, and alpha–adrenergic blocking agents are both vasodilators with blood-pressure-lowering effects. When vasodilators are used in combination, an additive effect on blood pressure may be anticipated. Clinical pharmacology studies have been conducted with coadministration of tadalafil with doxazosin, alfuzosin or tamsulosin [see Warnings and Precautions ( 5.1 ) and Clinical Pharmacology ( 12.2 )] .

 Antihypertensives 

 PDE5 inhibitors, including ADCIRCA, are mild systemic vasodilators. Clinical pharmacology studies were conducted to assess the effect of tadalafil on the potentiation of the blood–pressure–lowering effects of selected antihypertensive medications (amlodipine, angiotensin II receptor blockers, bendroflumethiazide, enalapril, and metoprolol). Small reductions in blood pressure occurred following coadministration of tadalafil with these agents compared with placebo [see Warnings and Precautions ( 5.1 ) and Clinical Pharmacology ( 12.2 )] .

 Alcohol 

 Both alcohol and tadalafil, a PDE5 inhibitor, act as mild vasodilators. When mild vasodilators are taken in combination, blood pressure–lowering effects of each individual compound may be increased. Substantial consumption of alcohol (e.g., 5 units or greater) in combination with ADCIRCA can increase the potential for orthostatic signs and symptoms, including increase in heart rate, decrease in standing blood pressure, dizziness, and headache. Tadalafil (10 mg or 20 mg) did not affect alcohol plasma concentrations and alcohol did not affect tadalafil plasma concentrations. [See Warnings and Precautions ( 5.1 ) and Clinical Pharmacology ( 12.2 )] .

 7.2 Potential for Other Drugs to Affect ADCIRCA

 Ritonavir 

 Ritonavir initially inhibits and later induces CYP3A, the enzyme involved in the metabolism of tadalafil. At steady state of ritonavir (about 1 week), the exposure to tadalafil is similar as in the absence of ritonavir [see Dosage and Administration ( 2.3 ), Warnings and Precautions ( 5.2 ), and Clinical Pharmacology ( 12.3 )] .

 Other Potent Inhibitors of CYP3A 

 Tadalafil is metabolized predominantly by CYP3A in the liver. In patients taking potent inhibitors of CYP3A such as ketoconazole, and itraconazole, avoid use of ADCIRCA [see Warnings and Precautions ( 5.2 ) and Clinical Pharmacology ( 12.3 )] .

 Potent Inducers of CYP3A 

 For patients chronically taking potent inducers of CYP3A, such as rifampin, avoid use of ADCIRCA [see Warnings and Precautions ( 5.2 ) and Clinical Pharmacology ( 12.3 )] .

 7.3 Potential for ADCIRCA to Affect Other Drugs

 Cytochrome P450 Substrates 

 Tadalafil is not expected to cause clinically significant inhibition or induction of the clearance of drugs metabolized by cytochrome P450 (CYP) isoforms (e.g., theophylline, warfarin, midazolam, lovastatin, bosentan) [see Clinical Pharmacology ( 12.3 )] .

 Aspirin 

 Tadalafil (10 mg and 20 mg once daily) does not potentiate the increase in bleeding time caused by aspirin [see Clinical Pharmacology ( 12.3 )] .

 P-glycoprotein

 (e.g., digoxin) 

 Coadministration of tadalafil (40 mg once daily) for 10 days did not significantly alter digoxin pharmacokinetics in healthy subjects [see Clinical Pharmacology ( 12.3 )] .<end_of_turn>
<start_of_turn>model
[{"type": "Unspecified interaction", "precipitant": "ritonavir", "precipitantCode": "N0000007423"}, {"type": "Unspecified interaction", "precipitant": "gc stimulator", "precipitantCode": "NO MAP"}, {"type": "Unspecified interaction", "precipitant": "riociguat", "precipitantCode": "N0000188995"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 25064002: Headache (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 28651003:  Orthostatic hypotension (disorder)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 404640003: Dizziness (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": "3424008: Tachycardia (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": "91081006: Vasodilatation (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alphaadrenergic blocking agents", "precipitantCode": "n0000175553", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alpha-blockers", "precipitantCode": "n0000029337", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "pde5 inhibitors", "precipitantCode": "N0000175599", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Unspecified interaction", "precipitant": "pde5 inhibitors", "precipitantCode": "N0000175599"}, {"type": "Unspecified interaction", "precipitant": "inducers of cyp3a", "precipitantCode": "n0000190118"}, {"type": "Unspecified interaction", "precipitant": "rifampin", "precipitantCode": "N0000006026"}, {"type": "Unspecified interaction", "precipitant": "nitrate", "precipitantCode": "N0000007647"}, {"type": "Pharmacodynamic interaction", "precipitant": "nitrates", "precipitantCode": "N0000007647", "effect": "45007003: Low blood pressure (disorder)"}, {"type": "Unspecified interaction", "precipitant": "nitrates", "precipitantCode": "N0000007647"}, {"type": "Unspecified interaction", "precipitant": "inhibitors of cyp3a", "precipitantCode": "N0000191001"}, {"type": "Unspecified interaction", "precipitant": "itraconazole", "precipitantCode": "N0000006753"}, {"type": "Unspecified interaction", "precipitant": "ketoconazole", "precipitantCode": "N0000007319"}, {"type": "Unspecified interaction", "precipitant": "organic nitrate", "precipitantCode": "NO MAP"}, {"type": "Pharmacodynamic interaction", "precipitant": "alpha adrenergic blocking agents", "precipitantCode": "N0000175553", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "vasodilators", "precipitantCode": "N0000175940", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "vasodilators", "precipitantCode": "N0000175940", "effect": "NO MAP"}, {"type": "Unspecified interaction", "precipitant": "cialis", "precipitantCode": "n0000010536"}, {"type": "Unspecified interaction", "precipitant": "cyp3a inhibitors", "precipitantCode": "N0000191001"}]<end_of_turn>
<start_of_turn>user
Based on the following prescribing information, provide the drug-drug interactions for the drug, Crestor, in a sturctured JSON list.

Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K).<end_of_turn>
<start_of_turn>model
[
```

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```json
[{"type": "Drug Interaction", "precipitant": "fluconazole", "effect": "Increased prothrombin time", "drug": "Warfarin"}, {"type": "Drug Interaction", "precipitant": "rifampin", "effect": "Decreased warfarin effectiveness", "drug": "Warfarin"}, {"type": "Drug Interaction", "precipitant": "leafy green vegetables", "effect": "Decreased warfarin effectiveness", "drug": "Warfarin"}]
```

## API drug information testing

### openFDA API

In [8]:
import requests
import json
from IPython.display import display, Markdown

def fetch_fda_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the openFDA API.
    """
    # API endpoint
    base_url = "https://api.fda.gov/drug/label.json"

    # Build query
    params = {
        'search': f'openfda.brand_name:"{drug_name}" openfda.generic_name:"{drug_name}"',
        'limit': 1
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            label = data['results'][0]
            # 'drug_interactions' is the standard field for Section 7
            raw_text = label.get('drug_interactions', ["Section 7 not found in this label."])[0]

            return {
                "drug": drug_name.upper(),
                "text": raw_text,
                "source": "openFDA API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "Epidiolex" # @param {type: "string"}
fetched = fetch_fda_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:500]}..."))

### API Data: EPIDIOLEX

**Raw Section 7 Snippet:**
> 7 DRUG INTERACTIONS • Strong inducer of CYP3A4 or CYP2C19: Consider dose increase of EPIDIOLEX. ( 7.1 ) • Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. ( 7.2 ) • A lower starting dose of orally administered everolimus is recommended. ( 7.2 ) • Consider dose modification of CYP2B6 or CYP2C19 substrates. ( 7.2 ) 7.1 Effect of Other Drugs on EPIDIOLEX Strong CYP3A4 or CYP2C19 Inducers Concomitant use with a strong CYP3A4 and CYP2C19 indu...

### DailyMed API

In [9]:
import requests
from xml.etree import ElementTree
from IPython.display import display, Markdown

def fetch_setId(drug_name):
    """
    Retrieves the SET ID for the first drug with the given name from the DailyMed API.
    """
    # API endpoint
    endpoint = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"

    # Build query
    params = {
        'drug_name': drug_name
    }

    try:
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        data = response.json()

        if 'data' in data:
            print(f"Found {len(data['data'])} results for {drug_name}.")
            print(f"Choosing index 0: {data['data'][0]['title']}")
            return data['data'][0]['setid']
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching SET ID for {drug_name}:** {str(e)}"))
        return None

def fetch_dailyMed_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the DailyMed API.
    """
    # Get the SET ID for the drug name
    setId = fetch_setId(drug_name)
    if setId == None:
      return None

    # API endpoint
    endpoint = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setId}.xml"

    try:
        response = requests.get(endpoint)
        response.raise_for_status()
        tree = ElementTree.fromstring(response.content)

        section = tree.find(".//{urn:hl7-org:v3}code[@code='34073-7']/..")

        if section is not None:
            return {
                "drug": drug_name.upper(),
                "text": "".join(section.itertext()).strip(),
                "source": "DailyMed API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "Epidiolex" # @param {type: "string"}
fetched = fetch_dailyMed_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:1000]}..."))

Found 1 results for Epidiolex.
Choosing index 0: EPIDIOLEX (CANNABIDIOL) SOLUTION [JAZZ PHARMACEUTICALS, INC.]


### API Data: EPIDIOLEX

**Raw Section 7 Snippet:**
> 7  DRUG INTERACTIONS
               
               
                  
                     
                        
                           
                              •Strong inducer of CYP3A4 or CYP2C19:  Consider dose increase of EPIDIOLEX. (7.1)
                           
                              •Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. (7.2)
                           
                              •A lower starting dose of orally administered everolimus is recommended. (7.2)
                           
                              •Consider dose modification of CYP2B6 or CYP2C19 substrates. (7.2)
                        
                     
                  
               
               
                  
                     
                     
                     7.1  Effect of Other Drugs on EPIDIOLEX
                     
                        
                           Strong CYP...

In [15]:
def test_new_drug_with_medgemma(drug_name, dailyMed=True):
    # Collect the FDA label
    collected_data = fetch_dailyMed_interactions(drug_name) if dailyMed else fetch_fda_interactions(drug_name)

    if not collected_data:
        display(Markdown(f"Could not find a label for **{drug_name}**"))
        return

    # Pass the collected text to our extraction engine
    display(Markdown(f"###Testing on: **{drug_name}**"))

    final_json = run_in_context_extraction(
        target_drug=collected_data['drug'],
        target_text=collected_data['text']
    )

    return final_json

# Try a drug that wasn't in the training set to see if it understands the pattern
test_drug = "Amlodipine" # @param {type: "string"}
use_dailyMed = True # @param {type: "boolean"}
test_case = test_new_drug_with_medgemma(test_drug)

Found 100 results for Amlodipine.
Choosing index 0: AMLODIPINE BESYLATE TABLET [PREFERRED PHARMACEUTICALS INC.]


###Testing on: **Amlodipine**

---

**The following text is being sent to the model:**

```text
<start_of_turn>user
You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

Based on the following prescribing information, provide the drug-drug interactions for the drug, Xarelto, in a sturctured JSON list.

TEXT OMITTED<end_of_turn>
<start_of_turn>model
[{"type": "Unspecified interaction", "precipitant": "ketoconazole", "precipitantCode": "N0000007319"}, {"type": "Pharmacodynamic interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacokinetic interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503", "effect": "C54355"}, {"type": "Unspecified interaction", "precipitant": "p-gp | inhibitors", "precipitantCode": "n0000185503"}, {"type": "Unspecified interaction", "precipitant": "ritonavir", "precipitantCode": "N0000007423"}, {"type": "Pharmacodynamic interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacokinetic interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141", "effect": "C54355"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 inhibitors", "precipitantCode": "n0000182141"}, {"type": "Unspecified interaction", "precipitant": "carbamazepine", "precipitantCode": "N0000007470"}, {"type": "Pharmacodynamic interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264", "effect": "371039008: Thromboembolic disorder (disorder)"}, {"type": "Pharmacokinetic interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264", "effect": "C54356"}, {"type": "Unspecified interaction", "precipitant": "p-gp | inducers", "precipitantCode": "n0000191264"}, {"type": "Unspecified interaction", "precipitant": "phenytoin", "precipitantCode": "N0000006023"}, {"type": "Unspecified interaction", "precipitant": "rifampin", "precipitantCode": "N0000006026"}, {"type": "Unspecified interaction", "precipitant": "st.   john 's   wort", "precipitantCode": "N0000022261"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 inducers", "precipitantCode": "n0000185506"}, {"type": "Pharmacodynamic interaction", "precipitant": "anticoagulants", "precipitantCode": "N0000029110", "effect": "131148009: Bleeding (finding)"}, {"type": "Unspecified interaction", "precipitant": "anticoagulants", "precipitantCode": "N0000029110"}, {"type": "Pharmacodynamic interaction", "precipitant": "aspirin", "precipitantCode": "N0000006582", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "chronic nsaid", "precipitantCode": "NO MAP", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "clopidogrel", "precipitantCode": "N0000007277", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "enoxaparin", "precipitantCode": "N0000006809", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "warfarin", "precipitantCode": "N0000006403", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "cyp3a4 inducers", "precipitantCode": "N0000185506", "effect": "371039008: Thromboembolic disorder (disorder)"}, {"type": "Pharmacokinetic interaction", "precipitant": "cyp3a4 inducers", "precipitantCode": "N0000185506", "effect": "C54356"}, {"type": "Unspecified interaction", "precipitant": "strong cyp3a4 | inducers", "precipitantCode": "n0000185506"}, {"type": "Unspecified interaction", "precipitant": "erythromycin", "precipitantCode": "N0000007133"}, {"type": "Unspecified interaction", "precipitant": "moderate cyp3a4 inhibitors", "precipitantCode": "n0000182141"}, {"type": "Pharmacodynamic interaction", "precipitant": "nsaids", "precipitantCode": "N0000175722", "effect": "131148009: Bleeding (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "platelet aggregation inhibitors", "precipitantCode": "N0000029299", "effect": "131148009: Bleeding (finding)"}]<end_of_turn>
<start_of_turn>user
Based on the following prescribing information, provide the drug-drug interactions for the drug, ADCIRCA, in a sturctured JSON list.

TEXT OMITTED<end_of_turn>
<start_of_turn>model
[{"type": "Unspecified interaction", "precipitant": "ritonavir", "precipitantCode": "N0000007423"}, {"type": "Unspecified interaction", "precipitant": "gc stimulator", "precipitantCode": "NO MAP"}, {"type": "Unspecified interaction", "precipitant": "riociguat", "precipitantCode": "N0000188995"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 25064002: Headache (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 28651003:  Orthostatic hypotension (disorder)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": " 404640003: Dizziness (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": "3424008: Tachycardia (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alcohol", "precipitantCode": "N0000007432", "effect": "91081006: Vasodilatation (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alphaadrenergic blocking agents", "precipitantCode": "n0000175553", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "alpha-blockers", "precipitantCode": "n0000029337", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "pde5 inhibitors", "precipitantCode": "N0000175599", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Unspecified interaction", "precipitant": "pde5 inhibitors", "precipitantCode": "N0000175599"}, {"type": "Unspecified interaction", "precipitant": "inducers of cyp3a", "precipitantCode": "n0000190118"}, {"type": "Unspecified interaction", "precipitant": "rifampin", "precipitantCode": "N0000006026"}, {"type": "Unspecified interaction", "precipitant": "nitrate", "precipitantCode": "N0000007647"}, {"type": "Pharmacodynamic interaction", "precipitant": "nitrates", "precipitantCode": "N0000007647", "effect": "45007003: Low blood pressure (disorder)"}, {"type": "Unspecified interaction", "precipitant": "nitrates", "precipitantCode": "N0000007647"}, {"type": "Unspecified interaction", "precipitant": "inhibitors of cyp3a", "precipitantCode": "N0000191001"}, {"type": "Unspecified interaction", "precipitant": "itraconazole", "precipitantCode": "N0000006753"}, {"type": "Unspecified interaction", "precipitant": "ketoconazole", "precipitantCode": "N0000007319"}, {"type": "Unspecified interaction", "precipitant": "organic nitrate", "precipitantCode": "NO MAP"}, {"type": "Pharmacodynamic interaction", "precipitant": "alpha adrenergic blocking agents", "precipitantCode": "N0000175553", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "vasodilators", "precipitantCode": "N0000175940", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "vasodilators", "precipitantCode": "N0000175940", "effect": "NO MAP"}, {"type": "Unspecified interaction", "precipitant": "cialis", "precipitantCode": "n0000010536"}, {"type": "Unspecified interaction", "precipitant": "cyp3a inhibitors", "precipitantCode": "N0000191001"}]<end_of_turn>
<start_of_turn>user
Based on the following prescribing information, provide the drug-drug interactions for the drug, AMLODIPINE, in a sturctured JSON list.

TEXT OMITTED<end_of_turn>
<start_of_turn>model
[
```

**MODEL RESPONSE:**

```json
[{"type": "Pharmacodynamic interaction", "precipitant": "CYP3A inhibitors", "precipitantCode": "N0000191001", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "CYP3A inducers", "precipitantCode": "N0000190118", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Unspecified interaction", "precipitant": "sildenafil", "precipitantCode": "N0000000000"}, {"type": "Pharmacodynamic interaction", "precipitant": "sildenafil", "precipitantCode": "N0000000000", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "simvastatin", "precipitantCode": "N0000000000", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "cyclosporine", "precipitantCode": "N0000000000", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Pharmacodynamic interaction", "precipitant": "tacrolimus", "precipitantCode": "N0000000000", "effect": "12763006: Finding of decreased blood pressure (finding)"}, {"type": "Unspecified interaction", "precipitant": "simvastatin", "precipitantCode": "N0000000000"}, {"type": "Unspecified interaction", "precipitant": "immunosuppressants", "precipitantCode": "N0000000000"}]
```